In [1]:
#packages
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
import math as m



In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/09 20:30:05 WARN Utils: Your hostname, Dions-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.86 instead (on interface en0)
25/10/09 20:30:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 20:30:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50797)
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/dionpapadopoulos/Library/Python/3.9/li

In [3]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    
    # === MEMORY MANAGEMENT ===
    .config("spark.driver.memory", "6g")          # Increase driver memory (safe for 16GB+ systems)
    .config("spark.executor.memory", "6g")        # Executors share same JVM locally
    .config("spark.driver.maxResultSize", "3g")   # Prevent large collect() results crashing driver

    # === PARALLELISM & SHUFFLING ===
    .config("spark.sql.shuffle.partitions", "64")  # More but smaller partitions for write stability
    .config("spark.default.parallelism", "8")      # ~ number of cores on your system
    .config("spark.sql.files.maxPartitionBytes", "64MB")  # Smaller file splits -> less peak mem per task

    # === PERFORMANCE TUNING ===
    .config("spark.memory.fraction", "0.85")        # 85% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.3")  # 30% of execution memory for caching
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # Fast pandas conversion

    # === WRITES / COMMIT ===
    .config("spark.sql.files.maxRecordsPerFile", "250000")  # cap rows per output file
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "false")  # avoid huge coalesced partitions on write

    # === TEMP STORAGE ===
    .config("spark.local.dir", "/tmp/spark-temp")   # Disk spill location for large shuffles

    # === DEFAULT OPTIONS ===
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", True)
    .config("spark.sql.session.timeZone", "Etc/UTC")

    .getOrCreate()
)

25/10/09 20:30:15 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
#reading in data

tbl_merchants_raw = spark.read.parquet('../data/tables/merchant_data/tbl_merchants.parquet')
consumer_user_details = spark.read.parquet('../data/tables/merchant_data/consumer_user_details.parquet')
transactions21 = spark.read.parquet('../data/tables/transaction_data/transactions_20210228_20210827_snapshot/')
transactions2122 = spark.read.parquet('../data/tables/transaction_data/transactions_20210828_20220227_snapshot/')
transactions22 = spark.read.parquet('../data/tables/transaction_data/transactions_20220228_20220828_snapshot/')
con_fraud_prob = spark.read.option("header","true").csv('../data/tables/merchant_data/consumer_fraud_probability.csv')
merch_fraud_prob = spark.read.option("header", "true").csv('../data/tables/merchant_data/merchant_fraud_probability.csv')

tbl_consumer_raw = spark.read.option("header", "true").csv('../data/tables/merchant_data/tbl_consumer.csv')

transactions = transactions21.unionByName(transactions2122)
transactions = transactions.unionByName(transactions22)

In [5]:
# Load first CSV
postcodes_df = spark.read.csv("../data/income/2024 Locality to 2021 SA2 Coding Index.csv", header=True, inferSchema=True)

# Load second CSV
income_df = spark.read.csv("../data/income/sa2_income.csv", header=True, inferSchema=True)

In [6]:
#Functions
def find_NULL(dfs):

    """Finds any rows with NULLs over different datasets"""

    for df in dfs:
        condition = f.lit(False)
        for col_name in df.columns:
            condition = condition | f.col(col_name).isNull()

        df.filter(condition).show()
    return df.filter(condition).count()

def filter_outliers(data, variables):
    
    """filters outliers of continuous data"""

    n=data.count()
    for feature in variables:
        # Calculate Q1 and Q3
        quantiles = data.approxQuantile(feature, [0.25, 0.75], 0.01)
        q1, q3 = quantiles
        iqr = q3 - q1

        #from ADS lecture slides, n>>100
        scale = m.sqrt(m.log(n)) - 0.5
        if scale<3:
            scale=3
        lower_bound = q1 - scale * iqr
        upper_bound = q3 + scale * iqr
        if lower_bound<0:
            data = data.filter((col(feature) >= 0) & (col(feature) <= upper_bound))
        else:
            data = data.filter((col(feature) >= lower_bound) & (col(feature) <= upper_bound))
    
    return data

def spark_shape(self):
    
    """Easy function for shape of a spark df"""

    return (self.count(), len(self.columns))

pyspark.sql.dataframe.DataFrame.shape = property(spark_shape)

In [7]:
#cleaning tags
string = "name|address|state|postcode|gender|consumer_id"

# Clean consumer table
tbl_consumer = (
    tbl_consumer_raw
    .withColumn("cust_name", f.split(col(string), "\\|").getItem(0))
    .withColumn("address", f.split(col(string), "\\|").getItem(1))
    .withColumn("state", f.split(col(string), "\\|").getItem(2))
    .withColumn("postcode", f.split(col(string), "\\|").getItem(3))
    .withColumn("gender", f.split(col(string), "\\|").getItem(4))
    .withColumn("consumer_id", f.split(col(string), "\\|").getItem(5))
    .drop(string)
)


# Clean merchants table
tbl_merchants = (
    tbl_merchants_raw
    # remove leading (( or [[ and trailing )) or ]]
    .withColumn(
        "tags_clean",
        f.regexp_replace(
            "tags",
            r"^\(\(|^\(\[|^\[\(|^\[\[|\)\)$|\]\)$|\)\]$|\]\]$",
            ""
        )
    )
    # split on `), (` or `], [`
    .withColumn("tags_array", f.split("tags_clean", r"\)\s*,\s*\(|\]\s*,\s*\["))
    # extract each element
    .withColumn("biz_tags", f.lower(f.col("tags_array")[0]))
    .withColumn("rev_band", f.col("tags_array")[1])
    .withColumn("take_rate", f.regexp_extract(f.col("tags_array")[2], r"take rate:\s*([0-9.]+)", 1)
    )
    .drop("tags", "tags_clean", "tags_array")
)

tbl_merchants=tbl_merchants.withColumn("biz_tags", f.regexp_replace("biz_tags", "  ", " "))

In [8]:
#joining transactions and merchants datasets
merchant_transactions=transactions.join(tbl_merchants, on='merchant_abn', how='left')
print(merchant_transactions.shape)
find_NULL([merchant_transactions])

(14195505, 9)


+------------+-------+------------------+--------------------+--------------+----+--------+--------+---------+
|merchant_abn|user_id|      dollar_value|            order_id|order_datetime|name|biz_tags|rev_band|take_rate|
+------------+-------+------------------+--------------------+--------------+----+--------+--------+---------+
| 29566626791|      8| 74.15732460440282|71a81652-cc91-4bf...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32234779638|  18490|107.14809429376949|20149572-a55b-41f...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 67202032418|     20| 55.46394975814555|a29071b4-29b3-4f2...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32461318592|     23| 613.9306657410166|4b2e2154-65d8-44f...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32234779638|     25| 87.15685629102919|e6763664-e95f-4eb...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 23633724513|     26|3459.2423030023524|fee9ead7-9ce2-4a4...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
|

580830

In [9]:
merchant_transactions = merchant_transactions.dropna()
print(merchant_transactions.shape)

(13614675, 9)


In [10]:
merchant_transactions.groupBy('name').count().orderBy("count", ascending=True).show()

+--------------------+-----+
|                name|count|
+--------------------+-----+
|    Curae Foundation|    1|
|Lobortis Nisi Ass...|    1|
|Elit Dictum Eu Fo...|    1|
|Aliquam Eu Institute|    1|
|Aenean Gravida In...|    1|
|       Phasellus LLP|    1|
|Consequat Foundation|    1|
|Dictum Mi Corpora...|    2|
|Egestas Nunc Sed LLC|    2|
|    Massa Rutrum LLP|    2|
|Accumsan Laoreet ...|    2|
|Tempor Augue Ac C...|    2|
|     Sem Corporation|    2|
|    Gravida Nunc LLP|    2|
|        Elit Limited|    2|
|Integer Urna Inst...|    2|
|            Elit LLP|    2|
|  Cras Convallis Ltd|    2|
|Semper Pretium Li...|    2|
|Adipiscing Fringi...|    2|
+--------------------+-----+
only showing top 20 rows


In [11]:
#filter outliers by biz_tag

n = merchant_transactions.count()
# compute the scale factor
scale = m.sqrt(m.log(n)) - 0.5
stats_by_band = (merchant_transactions.groupby('biz_tags')
                                      .agg(f.expr("percentile_approx(dollar_value, 0.25)").alias("Q1"),
                                           f.expr("percentile_approx(dollar_value, 0.75)").alias("Q3")
                ).withColumn("IQR", f.col("Q3") - f.col("Q1"))
                 .withColumn("lower_bound", f.col("Q1") - scale * f.col("IQR"))
                 .withColumn("upper_bound", f.col("Q3") + scale * f.col("IQR"))
                )
stats_by_band = stats_by_band.drop('IQR')

In [12]:
merchant_transactions = (
    merchant_transactions
    .join(stats_by_band, on="biz_tags", how="left")
    .filter(
        (col("dollar_value") >= col('lower_bound')) &
        (col("dollar_value") <= col("upper_bound"))
    )
    .select(merchant_transactions["*"])
)

merchant_transactions

merchant_abn,user_id,dollar_value,order_id,order_datetime,name,biz_tags,rev_band,take_rate
67609108741,18479,86.4040605836911,d0e180f0-cb06-42a...,2021-08-20,Metus Sit Amet In...,"cable, satellite,...",e,0.38
80508375382,3704,4.947605835919724,6748ba0b-9159-447...,2022-09-09,Pharetra Nibh Ins...,"cable, satellite,...",a,5.56
47663262928,9,36.69873283148887,c4fcb49a-ce87-4e1...,2021-08-20,Eget Lacus LLP,"cable, satellite,...",a,6.66
31101120643,3707,51.86995396992383,ee9e2523-d0be-49d...,2022-09-09,Commodo Hendrerit...,"cable, satellite,...",a,6.37
15299889494,23,69.00314523230432,f7c06643-1b8f-499...,2021-08-20,Porttitor Eros Ltd,"cable, satellite,...",b,5.01
16587082018,14829,142.43577656885788,be3c4273-c252-4ce...,2022-09-09,Laoreet Ipsum Ltd,"cable, satellite,...",a,6.29
10142254217,18511,70.72395057645588,14b037ad-466d-4e2...,2021-08-20,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22
31940875883,14834,43.80003102309123,f2f8acdc-e077-492...,2022-09-09,Amet Associates,"cable, satellite,...",a,6.46
29521780474,18516,68.74975650290615,3c6c21c3-35ae-4d0...,2021-08-20,At Sem Corp.,"cable, satellite,...",a,5.93
65668855732,14834,158.48831938258436,621cca03-1568-462...,2022-09-09,In Nec Industries,"cable, satellite,...",a,6.67


In [13]:
print(merchant_transactions.shape)

[154.711s][warning][gc,alloc] Executor task launch worker for task 36.0 in stage 60.0 (TID 1258): Retried waiting for GCLocker too often allocating 2915 words


(13293853, 9)


In [14]:
merchant_transactions=merchant_transactions.withColumnRenamed('name', 'business')
merchant_transactions=merchant_transactions.drop('order_id')
merchant_transactions

merchant_abn,user_id,dollar_value,order_datetime,business,biz_tags,rev_band,take_rate
67609108741,18479,86.4040605836911,2021-08-20,Metus Sit Amet In...,"cable, satellite,...",e,0.38
29521780474,16372,12.541022127303785,2022-05-08,At Sem Corp.,"cable, satellite,...",a,5.93
47663262928,9,36.69873283148887,2021-08-20,Eget Lacus LLP,"cable, satellite,...",a,6.66
31681476434,5321,132.1067173608104,2022-05-08,Lorem Sit LLC,"cable, satellite,...",c,2.22
15299889494,23,69.00314523230432,2021-08-20,Porttitor Eros Ltd,"cable, satellite,...",b,5.01
15299889494,5339,35.86248433330864,2022-05-08,Porttitor Eros Ltd,"cable, satellite,...",b,5.01
10142254217,18511,70.72395057645588,2021-08-20,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22
17488304283,16407,19.858236346093754,2022-05-08,Posuere Cubilia C...,"cable, satellite,...",a,6.18
29521780474,18516,68.74975650290615,2021-08-20,At Sem Corp.,"cable, satellite,...",a,5.93
94472466107,5346,15.403040697715044,2022-05-08,Eu Dolor Egestas PC,"cable, satellite,...",a,6.23


In [15]:
merchant_transactions.groupBy("biz_tags").agg(
    f.min("dollar_value").alias("min_value"),
    f.max("dollar_value").alias("max_value"),
    f.mean("dollar_value").alias("mean"),
    (f.max("dollar_value") - f.min("dollar_value")).alias("range")
).orderBy("mean", ascending=False).show()

[251.399s][warning][gc,alloc] Executor task launch worker for task 54.0 in stage 88.0 (TID 1634): Retried waiting for GCLocker too often allocating 13822 words
[251.406s][warning][gc,alloc] dag-scheduler-event-loop: Retried waiting for GCLocker too often allocating 331 words
[251.406s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 88.0 (TID 1617): Retried waiting for GCLocker too often allocating 15818 words
[251.406s][warning][gc,alloc] spark-listener-group-appStatus: Retried waiting for GCLocker too often allocating 332 words


[253.722s][warning][gc,alloc] Executor task launch worker for task 44.0 in stage 88.0 (TID 1667): Retried waiting for GCLocker too often allocating 5576 words
[253.780s][warning][gc,alloc] Executor task launch worker for task 20.0 in stage 88.0 (TID 1625): Retried waiting for GCLocker too often allocating 6087 words
[253.784s][warning][gc,alloc] Executor task launch worker for task 44.0 in stage 88.0 (TID 1667): Retried waiting for GCLocker too often allocating 5964 words


+--------------------+--------------------+------------------+------------------+------------------+
|            biz_tags|           min_value|         max_value|              mean|             range|
+--------------------+--------------------+------------------+------------------+------------------+
|jewelry, watch, c...|   3.409793978681009| 46001.13901942742| 9278.563185654188| 45997.72922544874|
|art dealers and g...|  0.4127496907944707| 10335.94618503865| 1966.235783927583|10335.533435347856|
|             telecom|  0.2931526313090789| 11606.18761084434|1735.6703991984127| 11605.89445821303|
|equipment, tool, ...|0.040595292090802974| 8813.127778854296| 1261.652706482699| 8813.087183562206|
|stationery, offic...|0.004010169952587961|2333.8271055469763|456.99076200821014| 2333.823095377024|
|health and beauty...| 6.59812930332817E-4|1672.2945523725923| 294.9553375878973| 1672.293892559662|
|motor vehicle sup...|7.092782606876731E-4|1356.3937038188778| 271.4403187831411| 1356.3929

In [16]:
biz_tags_list = merchant_transactions.select("biz_tags").distinct().rdd.flatMap(lambda x: x).collect()
print(biz_tags_list)
print(len(biz_tags_list))

['cable, satellite, and other pay television and radio services', 'tent and awning shops', 'gift, card, novelty, and souvenir shops', 'opticians, optical goods, and eyeglasses', 'equipment, tool, furniture, and appliance rent al and leasing', 'books, periodicals, and newspapers', 'antique shops - sales, repairs, and restoration services', 'hobby, toy and game shops', 'motor vehicle supplies and new parts', 'telecom', 'computer programming , data processing, and integrated systems design services', 'stationery, office supplies and printing and writing paper', 'lawn and garden supply outlets, including nurseries', 'music shops - musical instruments, pianos, and sheet music', 'computers, computer peripheral equipment, and software', 'watch, clock, and jewelry repair shops', 'artist supply and craft shops', 'shoe shops', 'florists supplies, nursery stock, and flowers', 'art dealers and galleries', 'furniture, home furnishings and equipment shops, and manufacturers, except appliances', 'bic

In [17]:
# Count how many distinct biz_tags each business has
biz_tag_counts = (
    merchant_transactions
    .groupBy("business")
    .agg(f.countDistinct("biz_tags").alias("distinct_tag_count"))
    .orderBy(f.desc("distinct_tag_count"))
)



In [18]:
# Assigning the biz_tags to segments
merchant_transactions = merchant_transactions.withColumn(
    "segment",
    f.when(f.col("biz_tags").isin(
        "watch, clock, and jewelry repair shops",
        "jewelry, watch, clock, and silverware shops",
        "shoe shops",
        "antique shops - sales, repairs, and restoration services",
        "gift, card, novelty, and souvenir shops"
    ), "Fashion, Jewelry & Personal Goods")
   
    .when(f.col("biz_tags").isin(
        "books, periodicals, and newspapers",
        "digital goods: books, movies, music",
        "music shops - musical instruments, pianos, and sheet music",
        "art dealers and galleries",
        "artist supply and craft shops",
        "hobby, toy and game shops",
        "cable, satellite, and other pay television and radio services"
    ), "Arts, Media & Entertainment")
   
    .when(f.col("biz_tags").isin(
        "computers, computer peripheral equipment, and software",
        "computer programming , data processing, and integrated systems design services",
        "telecom",
        "equipment, tool, furniture, and appliance rent al and leasing",
        "stationery, office supplies and printing and writing paper"
    ), "Technology & Professional Services")
   
    .when(f.col("biz_tags").isin(
        "furniture, home furnishings and equipment shops, and manufacturers, except appliances",
        "tent and awning shops",
        "lawn and garden supply outlets, including nurseries",
        "florists supplies, nursery stock, and flowers"
    ), "Home, Garden & Living")
   
    .when(f.col("biz_tags").isin(
        "opticians, optical goods, and eyeglasses",
        "health and beauty spas",
        "bicycle shops - sales and service",
        "motor vehicle supplies and new parts"
    ), "Lifestyle, Health & Recreation")
   
    .otherwise("Other")
)
merchant_transactions

merchant_abn,user_id,dollar_value,order_datetime,business,biz_tags,rev_band,take_rate,segment
67609108741,18479,86.4040605836911,2021-08-20,Metus Sit Amet In...,"cable, satellite,...",e,0.38,"Arts, Media & Ent..."
29521780474,16372,12.541022127303785,2022-05-08,At Sem Corp.,"cable, satellite,...",a,5.93,"Arts, Media & Ent..."
47663262928,9,36.69873283148887,2021-08-20,Eget Lacus LLP,"cable, satellite,...",a,6.66,"Arts, Media & Ent..."
31681476434,5321,132.1067173608104,2022-05-08,Lorem Sit LLC,"cable, satellite,...",c,2.22,"Arts, Media & Ent..."
15299889494,23,69.00314523230432,2021-08-20,Porttitor Eros Ltd,"cable, satellite,...",b,5.01,"Arts, Media & Ent..."
15299889494,5339,35.86248433330864,2022-05-08,Porttitor Eros Ltd,"cable, satellite,...",b,5.01,"Arts, Media & Ent..."
10142254217,18511,70.72395057645588,2021-08-20,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22,"Arts, Media & Ent..."
17488304283,16407,19.858236346093754,2022-05-08,Posuere Cubilia C...,"cable, satellite,...",a,6.18,"Arts, Media & Ent..."
29521780474,18516,68.74975650290615,2021-08-20,At Sem Corp.,"cable, satellite,...",a,5.93,"Arts, Media & Ent..."
94472466107,5346,15.403040697715044,2022-05-08,Eu Dolor Egestas PC,"cable, satellite,...",a,6.23,"Arts, Media & Ent..."


In [19]:
# Repartition to increase parallelism and reduce per-task memory pressure
# Rule of thumb: 2-3x number of cores or target ~128MB per partition after filter/join
num_partitions = int(spark.conf.get("spark.sql.shuffle.partitions"))
merchant_transactions.repartition(num_partitions, "biz_tags") \
    .write \
    .option("maxRecordsPerFile", spark.conf.get("spark.sql.files.maxRecordsPerFile")) \
    .mode("overwrite") \
    .parquet("../data/curated/merchant_transactions")

25/10/09 20:36:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/09 20:36:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/09 20:36:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [20]:
tbl_consumer=tbl_consumer.drop('address', 'cust_name', 'gender')
#tbl_consumer

In [21]:
mtc_fraud1=merchant_transactions.join(merch_fraud_prob, on=['merchant_abn', 'order_datetime'], how='left')
mtc_fraud1=mtc_fraud1.withColumnRenamed('fraud_probability', 'merch_fraud_prob')
mtc_fraud=mtc_fraud1.join(con_fraud_prob, on=['user_id', 'order_datetime'], how='left')
mtc_fraud=mtc_fraud.withColumnRenamed('fraud_probability', 'con_fraud_prob')
mtc_fraud

user_id,order_datetime,merchant_abn,dollar_value,business,biz_tags,rev_band,take_rate,segment,merch_fraud_prob,con_fraud_prob
18479,2021-08-20,67609108741,86.4040605836911,Metus Sit Amet In...,"cable, satellite,...",e,0.38,"Arts, Media & Ent...",NULL,NULL
23793,2022-05-02,24015173965,157.0,Lectus Limited,"cable, satellite,...",a,6.79,"Arts, Media & Ent...",NULL,NULL
9,2021-08-20,47663262928,36.69873283148887,Eget Lacus LLP,"cable, satellite,...",a,6.66,"Arts, Media & Ent...",NULL,NULL
23809,2022-05-02,49125619545,19.538619768814247,Adipiscing Elit C...,"cable, satellite,...",a,5.66,"Arts, Media & Ent...",NULL,NULL
23,2021-08-20,15299889494,69.00314523230432,Porttitor Eros Ltd,"cable, satellite,...",b,5.01,"Arts, Media & Ent...",NULL,NULL
23838,2022-05-02,82786382916,47.84461070001619,Eu Tempor Ltd,"cable, satellite,...",c,2.08,"Arts, Media & Ent...",NULL,NULL
18511,2021-08-20,10142254217,70.72395057645588,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22,"Arts, Media & Ent...",NULL,NULL
23842,2022-05-02,65204103269,157.0,Est Ac Mattis Ltd,"cable, satellite,...",b,4.93,"Arts, Media & Ent...",NULL,NULL
18516,2021-08-20,29521780474,68.74975650290615,At Sem Corp.,"cable, satellite,...",a,5.93,"Arts, Media & Ent...",NULL,NULL
23847,2022-05-02,69703285964,32.59955488451837,Suspendisse Incor...,"cable, satellite,...",a,5.77,"Arts, Media & Ent...",NULL,NULL


In [22]:
mtc_fraud

user_id,order_datetime,merchant_abn,dollar_value,business,biz_tags,rev_band,take_rate,segment,merch_fraud_prob,con_fraud_prob
18479,2021-08-20,67609108741,86.4040605836911,Metus Sit Amet In...,"cable, satellite,...",e,0.38,"Arts, Media & Ent...",NULL,NULL
23793,2022-05-02,24015173965,157.0,Lectus Limited,"cable, satellite,...",a,6.79,"Arts, Media & Ent...",NULL,NULL
9,2021-08-20,47663262928,36.69873283148887,Eget Lacus LLP,"cable, satellite,...",a,6.66,"Arts, Media & Ent...",NULL,NULL
23809,2022-05-02,49125619545,19.538619768814247,Adipiscing Elit C...,"cable, satellite,...",a,5.66,"Arts, Media & Ent...",NULL,NULL
23,2021-08-20,15299889494,69.00314523230432,Porttitor Eros Ltd,"cable, satellite,...",b,5.01,"Arts, Media & Ent...",NULL,NULL
23838,2022-05-02,82786382916,47.84461070001619,Eu Tempor Ltd,"cable, satellite,...",c,2.08,"Arts, Media & Ent...",NULL,NULL
18511,2021-08-20,10142254217,70.72395057645588,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22,"Arts, Media & Ent...",NULL,NULL
23842,2022-05-02,65204103269,157.0,Est Ac Mattis Ltd,"cable, satellite,...",b,4.93,"Arts, Media & Ent...",NULL,NULL
18516,2021-08-20,29521780474,68.74975650290615,At Sem Corp.,"cable, satellite,...",a,5.93,"Arts, Media & Ent...",NULL,NULL
23847,2022-05-02,69703285964,32.59955488451837,Suspendisse Incor...,"cable, satellite,...",a,5.77,"Arts, Media & Ent...",NULL,NULL


In [23]:
curated=mtc_fraud.groupBy(['merchant_abn', 'user_id']).agg(
    f.count('*').alias('count'),
    f.mean('dollar_value').alias('mean'),
    f.mean('merch_fraud_prob').alias('merch_fraud_prob'),
    f.mean('con_fraud_prob').alias('con_fraud_prob')
)
curated

[531.700s][warning][gc,alloc] Executor task launch worker for task 35.0 in stage 159.0 (TID 2646): Retried waiting for GCLocker too often allocating 493 words
[531.713s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 159.0 (TID 2634): Retried waiting for GCLocker too often allocating 19697 words


[581.369s][warning][gc,alloc] Executor task launch worker for task 63.0 in stage 171.0 (TID 2820): Retried waiting for GCLocker too often allocating 45064 words


25/10/09 20:39:48 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.


[584.851s][warning][gc,alloc] Executor task launch worker for task 63.0 in stage 171.0 (TID 2820): Retried waiting for GCLocker too often allocating 2097154 words


25/10/09 20:39:54 WARN TaskMemoryManager: Failed to allocate a page (67108864 bytes), try again.


merchant_abn,user_id,count,mean,merch_fraud_prob,con_fraud_prob
40252040480,18574,1,110.20656881653261,NULL,NULL
65668855732,18702,1,89.11540693789668,NULL,NULL
65668855732,10420,3,34.063783533774306,NULL,NULL
20885454195,10480,1,151.10448160559477,NULL,NULL
17488304283,10651,3,80.03352581967776,NULL,NULL
14430838529,11174,1,38.63463291681361,NULL,NULL
21439773999,11182,8,61.365044043924165,NULL,NULL
17488304283,1954,3,69.99194182081949,NULL,NULL
41974958954,20742,3,85.63999694233401,NULL,NULL
38523562929,2402,2,80.46693259606917,NULL,NULL


In [24]:
new_curated=curated.join(consumer_user_details, on='user_id', how='left')
new_curated=new_curated.join(tbl_consumer, on='consumer_id', how='left')
new_curated=new_curated.drop('consumer_id')
new_curated

[619.085s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 185.0 (TID 2977): Retried waiting for GCLocker too often allocating 18777 words


[642.828s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 185.0 (TID 2976): Retried waiting for GCLocker too often allocating 7200731 words


25/10/09 20:40:46 WARN TaskMemoryManager: Failed to allocate a page (57605825 bytes), try again.


[675.028s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 205.0 (TID 3217): Retried waiting for GCLocker too often allocating 65538 words
[675.048s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 205.0 (TID 3216): Retried waiting for GCLocker too often allocating 20512 words


25/10/09 20:41:18 ERROR Executor: Exception in task 2.0 in stage 205.0 (TID 3217)
java.lang.OutOfMemoryError: Java heap space
25/10/09 20:41:18 ERROR SparkUncaughtExceptionHandler: Uncaught exception in thread Thread[Executor task launch worker for task 2.0 in stage 205.0 (TID 3217),5,main]
java.lang.OutOfMemoryError: Java heap space
25/10/09 20:41:19 ERROR Inbox: Ignoring error
java.util.concurrent.RejectedExecutionException: Task org.apache.spark.executor.Executor$TaskRunner@5d9f759e rejected from java.util.concurrent.ThreadPoolExecutor@2e18a29c[Shutting down, pool size = 12, active threads = 12, queued tasks = 0, completed tasks = 3220]
	at java.base/java.util.concurrent.ThreadPoolExecutor$AbortPolicy.rejectedExecution(ThreadPoolExecutor.java:2065)
	at java.base/java.util.concurrent.ThreadPoolExecutor.reject(ThreadPoolExecutor.java:833)
	at java.base/java.util.concurrent.ThreadPoolExecutor.execute(ThreadPoolExecutor.java:1365)
	at org.apache.spark.executor.Executor.launchTask(Execut

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
print(new_curated.shape)

In [ ]:
new_curated.write.parquet("data/curated/agg_by_userbiz", mode="overwrite")

In [ ]:
# Rename columns in df2 for easier handling
income_clean = (
    income_df
    .withColumnRenamed("Statistical Areas Level 2 2021 code", "SA2_CODE_2021")
    .withColumnRenamed("Statistical Areas Level 2 2021 name", "SA2_NAME_2021")
)

# Make sure join keys are the same type
postcodes_df = postcodes_df.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`") != 0)

# Perform join on SA2 code
merged_df = postcodes_df.join(income_clean, on="SA2_CODE_2021", how="left")
merged_df = merged_df.drop(income_clean.SA2_NAME_2021)  # drop df2’s version

missing_count = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNull()).count()
print(missing_count)

# Dropping NULL values in income column
print(merged_df.count()) # count before dropping NULL values
merged_df = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNotNull() )
print(merged_df.count()) # count after dropping NULL values

result_df = (
    merged_df
    .groupBy(col("POSTCODE").alias("postcode"))
    .agg(
        f.avg(
            col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`")
        ).alias("median_total_income_2020")
    )
)
result_df

In [ ]:
result_df = result_df.withColumn(
    "income_bin",
    f.when(f.col("median_total_income_2020") < 30000, "<30k")
     .when((f.col("median_total_income_2020") >= 30000) & (f.col("median_total_income_2020") < 40000), "30-40k")
     .when((f.col("median_total_income_2020") >= 40000) & (f.col("median_total_income_2020") < 50000), "40-50k")
     .when((f.col("median_total_income_2020") >= 50000) & (f.col("median_total_income_2020") < 60000), "50-60k")
     .when((f.col("median_total_income_2020") >= 60000) & (f.col("median_total_income_2020") < 70000), "60-70k")
     .when((f.col("median_total_income_2020") >= 70000) & (f.col("median_total_income_2020") < 80000), "70-80k")
     .otherwise("80k+")
)

result_df.show(100, truncate=False)
result_df.printSchema()


In [ ]:
result_df.write.csv("../data/curated/merged_postcode_income.csv", header=True, mode="overwrite")